# Mini GPT-2 (decoder-only transformer) trained from scratch on Tiny Shakespeare

This notebook builds and trains a small, GPT-2-style decoder-only transformer entirely from scratch (no `transformers` library, no pretrained weights) — following the same architecture we've been walking through: character-level tokenization, causal self-attention, multi-head attention, residual connections, and pre-LayerNorm blocks.

**Hyperparameters below match the standard "Let's build GPT" recipe (~10M parameters):**
- 6 layers, 6 heads, 384 embedding dim, 256 context length, 0.2 dropout, 3e-4 learning rate

**No manual dataset download needed** — the code below fetches Tiny Shakespeare (~1MB) directly from GitHub.

Run cells top to bottom. On Colab: `Runtime → Change runtime type → T4 GPU` before you start.

In [1]:
# Confirm which GPU Colab assigned (and that one was assigned at all)
!nvidia-smi

Fri Jun 26 18:04:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
import math
import urllib.request

import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

In [3]:
# ----- Hyperparameters -----
batch_size    = 64       # independent sequences processed in parallel
block_size    = 256      # context length (max tokens model can attend back to)
max_iters     = 5000      # total training steps
eval_interval = 500       # how often to report train/val loss + checkpoint
eval_iters    = 200        # batches averaged over when estimating loss
learning_rate = 3e-4
n_embd        = 384
n_head        = 6
n_layer       = 6
dropout       = 0.2

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cpu':
    print('No GPU detected — this will be slow. In Colab: Runtime > Change runtime type > T4 GPU')

device: cuda


### Optional: persist checkpoints to Google Drive

Colab's free tier can disconnect a session without warning. If `USE_DRIVE = True`, checkpoints survive a disconnect and training can resume where it left off (the training cell below already supports resuming). Leave `False` for a quick run where you don't care about persistence.

In [4]:
USE_DRIVE = False  # set True to checkpoint to Google Drive instead of the local (ephemeral) disk

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    checkpoint_dir = '/content/drive/MyDrive/mini_gpt2'
    os.makedirs(checkpoint_dir, exist_ok=True)
else:
    checkpoint_dir = '.'

checkpoint_path = os.path.join(checkpoint_dir, 'mini_gpt2_checkpoint.pt')
print('checkpoints will be saved to:', checkpoint_path)

checkpoints will be saved to: ./mini_gpt2_checkpoint.pt


## Step 1 — Dataset

Downloads automatically — nothing to upload or fetch yourself. This is the same Tiny Shakespeare file Karpathy uses in the lecture, hosted on GitHub.

In [5]:
input_file_path = 'input.txt'
if not os.path.exists(input_file_path):
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    urllib.request.urlretrieve(url, input_file_path)
    print('downloaded', url)
else:
    print('already downloaded, reusing local copy')

with open(input_file_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(f'dataset length: {len(text):,} characters')
print(text[:300])

downloaded https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
dataset length: 1,115,394 characters
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


## Step 2 — Tokenization (character-level)

Every unique character in the file becomes one token. Simple, no merges to learn, and it's exactly what lets us validate the model architecture before introducing a real subword tokenizer later.

In [6]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

print(f'vocab size: {vocab_size}')
print(''.join(chars))

vocab size: 65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [7]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f'train tokens: {len(train_data):,}, val tokens: {len(val_data):,}')

train tokens: 1,003,854, val tokens: 111,540


## Step 3 — Batching

Pulls `batch_size` random windows of length `block_size` from the data, each paired with the same window shifted one character to the right — that shift is the next-token target at every position.

In [8]:
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

## Step 4 — Model

Built bottom-up: a single causal attention head, multi-head attention (heads run in parallel and concatenate), a feed-forward block, then the full pre-LayerNorm transformer block: `x = x + Attn(LN(x))`, `x = x + FFN(LN(x))`.

In [9]:
class Head(nn.Module):
    """One head of causal self-attention."""

    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # lower-triangular mask: position i can only see positions <= i
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)        # (B, T, T) scaled scores
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # causal mask
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    """Several causal attention heads in parallel, concatenated and projected back."""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    """Per-token MLP: expand 4x, nonlinearity, project back down.

    Uses ReLU to match the lecture exactly. Real GPT-2 uses GELU instead —
    swap nn.ReLU() for nn.GELU() below if you want the closer-to-GPT-2 variant.
    """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """One transformer block: pre-LayerNorm attention, then pre-LayerNorm MLP, each with a residual add."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [10]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                       # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # (T, n_embd)
        x = tok_emb + pos_emb                                           # broadcast over batch
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                        # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]               # crop to last block_size tokens
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]                      # last time step only
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


model = GPTLanguageModel().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'{n_params/1e6:.2f}M parameters')

10.79M parameters


## Step 5 — Training

`estimate_loss()` averages loss over several batches so the printed number isn't noisy from a single lucky/unlucky batch. The training loop below checkpoints every `eval_interval` steps and will **resume automatically** if a checkpoint already exists at `checkpoint_path` — re-running this cell after a disconnect picks up where you left off instead of starting over.

In [11]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [12]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

start_iter = 0
if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_iter = ckpt['iter'] + 1
    print(f'resumed from checkpoint at iter {start_iter}')

for it in range(start_iter, max_iters):
    if it % eval_interval == 0 or it == max_iters - 1:
        losses = estimate_loss()
        print(f"step {it}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'iter': it,
        }, checkpoint_path)

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print('training complete')

step 0: train loss 4.2221, val loss 4.2306
step 500: train loss 1.7605, val loss 1.9163
step 1000: train loss 1.3937, val loss 1.6050
step 1500: train loss 1.2649, val loss 1.5270
step 2000: train loss 1.1852, val loss 1.5007
step 2500: train loss 1.1220, val loss 1.4850
step 3000: train loss 1.0730, val loss 1.4853
step 3500: train loss 1.0186, val loss 1.5067
step 4000: train loss 0.9617, val loss 1.5093
step 4500: train loss 0.9116, val loss 1.5376
step 4999: train loss 0.8561, val loss 1.5513
training complete


## Step 6 — Generate text

Starts from a single newline character and samples forward. Expect genuinely Shakespeare-*shaped* text (character names, line structure, archaic words) rather than coherent sense — that's the correct outcome for a 10M-parameter character-level model at this loss range (val loss should be in the 1.4–1.6 area with the hyperparameters above).

In [13]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=500)[0].tolist()
print(decode(generated))


But with painted and guishest wind toward
Of Polamas as shed lions, and father,
It will as trous as over mutine,
We, no more honour, none would end more silver foot
For weeping: to whom, they save mere a little,--
This ripwly seems to the bleed pock:
We want her maon! We'll addle to-day.
By county for Montague,
That dost thinkingly love? Welcome, my lord.
Welcome, the king that not resemble;
How! what news well'd in her help before,
And in this, I had seen his time in recoion?
Who mufflet goes a


## Where to go from here

This is the validated baseline. The bonus directions from the project brief build on this exact same code:
- **Multilingual**: swap the character tokenizer for a byte-level or multilingual BPE tokenizer (e.g. `tiktoken`, or a custom `sentencepiece`/`tokenizers` BPE) and point `text` at a multilingual corpus. No architecture change needed.
- **Real subword mini-GPT2**: same model classes, swap in a BPE tokenizer (a few thousand tokens) and a larger corpus (WikiText-103 or a slice of OpenWebText).
- **Full 124–127M GPT-2 scale**: bump `n_layer=12, n_head=12, n_embd=768, block_size=1024`, switch `FeedForward` to `nn.GELU()`, tie `lm_head.weight = token_embedding_table.weight`, and add `torch.cuda.amp.autocast` + `GradScaler` for mixed precision — you'll need gradient accumulation to keep the effective batch size reasonable in 16GB.